# Track 10 — Capstone: Personal Agent (개인 비서 에이전트 · Ledger + Skills)

## Memory Ledger + Skills Manifest란?

개인 비서 에이전트는 사용자의 요청을 기억하고 필요한 능력을 골라 실행해야 합니다. 이 노트북은 그 최소 골격을 두 요소로 나눠 보여줍니다.

- **Memory Ledger:** 사용자 질의와 도구 결과를 시간순으로 쌓는 append-only 기록입니다. 저장·복원하면 세션을 넘어 이어집니다.
- **Skills manifest:** `calendar`와 `reminder` 두 skill을 함수 스키마와 실행 콜백으로 등록한 도구 목록입니다.
- **패키지:** ledger, skills, 정적 회귀 결과를 하나의 JSON으로 묶습니다.

> **hermes minimal:** [`implementations/hermes-agent`](../../implementations/hermes-agent)의 최소 골격판입니다. 라이브 LLM과 외부 연동 없이 기억·능력 루프만 결정론적으로 보여줍니다.

## 이 노트북에서 보여줄 것

| Session | 보여주는 것 | 목적 |
|---|---|---|
| 1. Setup | facade 시작 · 골든 로더 · 정적 메트릭 데모 · 패키지 저장 헬퍼 | 공통 준비 |
| 2. Ledger + skills | skill 등록 → 실행 → ledger 기록·재생 → JSONL 저장·복원·재활용 | 개인 에이전트의 기억과 능력 흐름 확인 |
| 3. 패키지 | 골든 회귀 + ledger·skills·trace 저장 | 제출·회귀 산출물 마감 |

## 이 노트북을 마치면

- `calendar`/`reminder` skill을 등록하고 질의별로 실행할 수 있습니다.
- 질의와 도구 결과를 ledger에 쌓고, JSONL로 저장·복원해 다음 행동에 재활용할 수 있습니다.
- ledger·skills·정적 메트릭을 단일 패키지로 묶을 수 있습니다.

**산출물:** `_out/06/capstone_package.json`, `_out/06/ledger.jsonl`  
**실행 조건:** 전 구간 API 키 없이 실행됩니다.

> 요약: Memory Ledger와 두 skill로 개인 비서의 최소 골격을 만들고, 저장·복원·재활용까지 확인하는 캡스톤입니다.


## Session 1. Setup


### Session 1-1. Setup

**하는 일:** 경로·클라이언트·공통 헬퍼를 준비합니다.

**정상:** `exaone 0.1.0 | model LGAI-EXAONE/K-EXAONE-236B-A23B | HAS_API True` 형태로 버전·모델·키 보유 여부 한 줄이 출력됩니다. 이어서 `load_capstone_golden`·`regression_m1_m6_m9`·`save_package` 헬퍼가 정의됩니다(출력 없음).

**의미:** 이후 단계에서 쓸 경로·클라이언트가 맞는지 먼저 봅니다.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import logging

# (en) Quiet library logs so the notebook output stays readable.
# (kr) 라이브러리 로그를 줄여 노트북 출력을 읽기 쉽게 한다.
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)

# (en) Facade-only startup; requires editable install at the repo root.
# (kr) `exaone` facade로 시작한다. 저장소 루트에서 editable 설치가 필요하다.
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "`exaone`이 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
ROOT = exaone.project_root()
TRACK10 = ROOT / "recipes" / "track10_ax_capstones"
DATA = TRACK10 / "data"
API_KEY = os.environ.get("EXAONE_API_KEY", "").strip()
BASE_URL = os.environ.get("EXAONE_BASE_URL", "").strip() or "http://localhost:8000/v1"
MODEL = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
HAS_API = bool(API_KEY)
client = None
if HAS_API:
    client = exaone.llm.ExaoneAPIClient(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
print("exaone", exaone.__version__, "| model", MODEL, "| HAS_API", HAS_API)

def load_capstone_golden(tag: str) -> list[dict]:
    # (en) Load golden rows for this capstone id or shared "all" rows.
    # (kr) 해당 캡스톤과 공통 "all" 골든 사례를 함께 불러온다.
    rows: list[dict] = []
    for line in (DATA / "capstone_golden.jsonl").read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if row.get("capstone") in (tag, "all"):
            rows.append(row)
    return rows


def regression_m1_m6_m9(rows: list[dict]) -> dict:
    # (en) Static fixture metric demo (M1/M6/M9) over golden rows — metric MECHANICS, not live agent perf.
    # (kr) 정적 골든 fixture로 M1/M6/M9를 계산한다. 라이브 에이전트 성능이 아니라 메트릭 동작 예시다.
    from eval.metrics import m1_task_success, m6_schema_adherence
    from eval.metrics.m1_task_success import TaskGold
    from eval.metrics.m6_schema_adherence import SchemaSpec
    from eval.metrics.m9_faithfulness import LengthRatioJudge
    from eval.metrics.types import TrialResult

    m1s, m6s, m9s = [], [], []
    cases = []
    for row in rows:
        tid = row["id"]
        content = row.get("trial_content") or str(row.get("expected_answer", ""))
        tr = TrialResult(
            trial_id=f"cap-{tid}",
            task_id=tid,
            dataset="track10.golden",
            runner="capstone",
            final_content=content,
        )
        m1 = m6 = m9 = None
        if row.get("expected_answer") is not None:
            m1 = m1_task_success.score_trial_exact(tr, TaskGold(task_id=tid, answer=row["expected_answer"]))
            m1s.append(m1)
        rk = row.get("required_keys")
        if rk:
            _, loose = m6_schema_adherence.score_trial(tr, SchemaSpec(required_keys=rk))
            m6 = loose
            m6s.append(1.0 if loose else 0.0)
        if row.get("grounding_context"):
            m9 = LengthRatioJudge()(trial=tr, gold={"context": row["grounding_context"]})
            m9s.append(m9)
        cases.append({"id": tid, "M1": m1, "M6": m6, "M9": m9})
    mean = lambda xs: sum(xs) / len(xs) if xs else 0.0
    return {"n": len(rows), "M1_mean": mean(m1s), "M6_loose_mean": mean(m6s), "M9_mean": mean(m9s), "cases": cases}


def save_package(capstone_nb: str, body: dict) -> Path:
    # (en) Write capstone_package.json under <track>/_out/<nb>/ (absolute path, CWD-independent).
    # (kr) 절대경로로 <track>/_out/<nb>/capstone_package.json을 저장한다(커널 CWD와 무관).
    out_dir = TRACK10 / "_out" / capstone_nb
    out_dir.mkdir(parents=True, exist_ok=True)
    slo = exaone.observability.SLOSpec(
        name=f"capstone-{capstone_nb}",
        p95_chat_latency_ms=8000,
        structured_output_success_min="95%",
        notes="Track 10 capstone — adjust per deployment.",
    )
    payload = {
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "capstone_id": capstone_nb,
        "slo": slo.to_dict(),
        **body,
    }
    path = out_dir / "capstone_package.json"
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved", path.resolve())
    return path


**출력 해석:** `exaone 0.1.0 | model LGAI-EXAONE/K-EXAONE-236B-A23B | HAS_API True` 한 줄로 이 노트북의 공통 토대가 준비되었음을 확인합니다.

- `model LGAI-EXAONE/K-EXAONE-236B-A23B`는 `.env`의 `EXAONE_MODEL` 값으로, 라이브 호출이 아니라 패키지 SLOSpec·표시용으로만 쓰입니다 — 이 캡스톤은 ledger·skills·메트릭이 모두 결정론이라 모델을 실제로 부르지 않습니다.
- `HAS_API True`는 키가 설정되어 있다는 표시일 뿐이며, 키가 없어도(`False`) 전 구간이 그대로 동작합니다 — `client` 만 `None` 으로 남고 분기에 영향을 주지 않습니다.
- 헬퍼 3종(`load_capstone_golden`/`regression_m1_m6_m9`/`save_package`)은 정의만 되고 아직 실행되지 않아 출력이 없습니다 — Session 3 에서 호출됩니다.


## Session 2. Ledger + skills


### Session 2-1. Ledger + skills

**하는 일:** 두 skill을 **하나의 레지스트리**에 등록한 뒤, 실제 질의에 대해 skill을 골라 **실행**하고 그 결과를 ledger 에 기록하고, 마지막으로 ledger를 **다시 읽어** 무엇이 기억되었는지 확인합니다.

**정상:** 네 부분이 출력됩니다 — ① `manifest: ['calendar', 'reminder']`(등록된 두 skill), ② 질의별 `[질의] -> skill -> 실제 반환값` 두 줄, ③ `ledger len 4`, ④ ledger 재생 4줄(`user_query`/`tool_result` 교대).

**의미:** 질의 → 능력 선택·실행 → 결과 기억 → 재생까지 한 흐름으로 이어져, 개인 비서의 '기억(ledger)'과 '능력(skills)'이 실제로 어떻게 맞물리는지 눈으로 따라갑니다.


In [ ]:
# (en) Memory ledger = append-only event log; the agent's working memory.
# (kr) Memory ledger = append-only 이벤트 로그; 에이전트의 작업 기억.
ledger = exaone.memory.InMemoryLedger(max_entries=100)


def skill_calendar(_n: str, args: dict) -> dict:
    # (en) Stub skill: return today's single fixed event (no real calendar).
    # (kr) 스텁 skill: 오늘의 고정 일정 1건을 반환한다(실제 캘린더 아님).
    return exaone.tools.ToolResult.success(content="2026-05-28 14:00 팀 싱크", source="calendar").to_dict()


def skill_reminder(_n: str, args: dict) -> dict:
    # (en) Stub skill: echo back the reminder text it was asked to set.
    # (kr) 스텁 skill: 등록 요청받은 리마인더 문구를 그대로 돌려준다.
    return exaone.tools.ToolResult.success(content=f"reminder set: {args.get('text','')}", source="reminder").to_dict()


# (en) ONE registry holds BOTH skills — a single manifest the agent picks from.
# (kr) 하나의 레지스트리에 두 skill을 모은다 — 에이전트가 골라 쓰는 단일 매니페스트.
reg = exaone.tools.ToolRegistry()
for schema, fn, name in [
    ({"type": "function", "function": {"name": "calendar", "description": "today", "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}}, skill_calendar, "calendar"),
    ({"type": "function", "function": {"name": "reminder", "description": "set reminder", "parameters": {"type": "object", "required": ["text"], "properties": {"text": {"type": "string"}}, "additionalProperties": False}}}, skill_reminder, "reminder"),
]:
    reg.register(exaone.tools.tool_from_callable(name, schema, fn))
print("manifest:", [s["function"]["name"] for s in reg.get_schemas()])

# (en) For each query: record it, pick & EXECUTE the matching skill, record the result.
# (kr) 각 질의마다: 기록하고, 맞는 skill을 골라 실행하고, 결과를 기록한다.
for query, tool, tool_args in [
    ("내일 일정 알려줘", "calendar", {}),
    ("회의 자료 준비 리마인더 걸어줘", "reminder", {"text": "회의 자료 준비"}),
]:
    ledger.append(event_type="user_query", hint=query)
    result = reg.execute(tool, tool_args)
    ledger.append(event_type="tool_result", hint=result["content"])
    print(f"[{query}] -> {tool} -> {result['content']}")

# (en) Read the ledger BACK — what the agent now remembers, replayable as context.
# (kr) ledger를 다시 읽는다 — 에이전트가 기억하는 내용, 맥락으로 재생 가능.
print("ledger len", len(ledger))
for entry in ledger.as_list():
    print(f"  - {entry.event_type}: {entry.hint}")


**출력 해석:** `manifest` 부터 `ledger len 4` · 재생 목록까지, 질의가 능력을 거쳐 기억으로 누적되는 한 사이클이 그대로 드러납니다.

- `manifest: ['calendar', 'reminder']` — 두 skill 이 **하나의 레지스트리**에 모여 에이전트가 골라 쓸 도구 목록이 됩니다(이전처럼 루프마다 레지스트리를 새로 만들어 마지막 하나만 남기지 않습니다).
- `[내일 일정 알려줘] -> calendar -> 2026-05-28 14:00 팀 싱크` / `[회의 자료 준비 리마인더 걸어줘] -> reminder -> reminder set: 회의 자료 준비` — 질의에 맞는 skill을 골라 `reg.execute(...)` 로 **실제 실행**한 결과입니다. 반환값은 라이브 LLM 이 아닌 **고정 콜백**이라 매번 결정론적입니다.
- `ledger len 4` — 질의 2건 × (`user_query` + `tool_result`) = 이벤트 4개가 append-only 로 쌓였습니다. 이어지는 재생 목록은 ledger를 `as_list()` 로 **다시 읽은** 것으로, 에이전트가 "방금 무슨 일이 있었는지"(질의와 그 실행 결과)를 순서대로 보존함을 보여줍니다 — 이 기억이 다음 턴의 맥락으로 재사용될 수 있습니다.


### Session 2-2. 저장 → 복원 → 재활용 (cross-session memory)

**하는 일:** Session 2-1 의 ledger를 `_out/06/ledger.jsonl` 로 **저장**한 뒤, "새 세션"처럼 빈 ledger 에 **복원**하고, 복원된 기억(직전 일정)을 읽어 **다음 리마인더에 실제로 사용**합니다.

**정상:** `saved ledger: …/ledger.jsonl | entries: 4` → `resumed session | ledger entries: 4` → `recalled schedule: 2026-05-28 14:00 팀 싱크` → `[방금 조회한 일정에 리마인더 걸어줘] -> reminder -> reminder set: 2026-05-28 14:00 팀 싱크 준비` → `ledger len 6` + 재생 6줄.

**의미:** ledger 가 디스크를 거쳐 **세션을 넘어 살아남고**, 그 기억이 다음 행동을 바꾸는 것을 보여줍니다 — '기억'이 단순 기록을 넘어 쓸모를 갖는 지점입니다.


In [ ]:
# (en) Persist the ledger as JSONL — Track 05's snapshot/restore idea, ledger-only, no key needed.
# (kr) ledger를 JSONL 로 저장 — Track 05 의 스냅샷/복원 아이디어를 ledger 한정으로, 키 불필요.
LEDGER_PATH = TRACK10 / "_out" / "06" / "ledger.jsonl"
LEDGER_PATH.parent.mkdir(parents=True, exist_ok=True)
LEDGER_PATH.write_text(
    "\n".join(json.dumps({"event_type": e.event_type, "hint": e.hint}, ensure_ascii=False) for e in ledger.as_list()) + "\n",
    encoding="utf-8",
)
print("saved ledger:", LEDGER_PATH.resolve(), "| entries:", len(ledger))

# (en) NEW session: a fresh empty ledger that RESTORES prior memory from disk.
# (kr) 새 세션: 빈 ledger를 만들어 디스크에서 직전 기억을 복원한다.
ledger2 = exaone.memory.InMemoryLedger(max_entries=100)
for line in LEDGER_PATH.read_text(encoding="utf-8").splitlines():
    if line.strip():
        row = json.loads(line)
        ledger2.append(event_type=row["event_type"], hint=row["hint"])
print("resumed session | ledger entries:", len(ledger2))

# (en) Recall the calendar result from restored memory and USE it in the next action.
# (kr) 복원된 기억에서 캘린더 결과를 꺼내 다음 행동에 실제로 사용한다.
schedule = next(e.hint for e in ledger2.as_list() if e.event_type == "tool_result")
print("recalled schedule:", schedule)
followup = "방금 조회한 일정에 리마인더 걸어줘"
ledger2.append(event_type="user_query", hint=followup)
result = reg.execute("reminder", {"text": f"{schedule} 준비"})
ledger2.append(event_type="tool_result", hint=result["content"])
print(f"[{followup}] -> reminder -> {result['content']}")
print("ledger len", len(ledger2))
for entry in ledger2.as_list():
    print(f"  - {entry.event_type}: {entry.hint}")


**출력 해석:** 저장 → 복원 → 재활용으로, ledger 가 **한 번의 실행 안에서만 사는 게 아니라** 세션 경계를 넘어 기억으로 작동함을 보여줍니다.

- `saved ledger: … | entries: 4` / `resumed session | ledger entries: 4` — Session 2-1 의 4개 이벤트를 JSONL 로 내보냈다가 **빈 새 ledger 로 그대로 복원**했습니다. 같은 길이(4)가 복원이 손실 없이 됐다는 증거입니다.
- `recalled schedule: 2026-05-28 14:00 팀 싱크` — 복원된 기억에서 **직전 캘린더 결과를 꺼냈습니다**. 이 값은 코드에 다시 적은 게 아니라 disk → ledger를 거쳐 돌아온 것입니다.
- `[방금 조회한 일정에 리마인더 걸어줘] -> reminder -> reminder set: 2026-05-28 14:00 팀 싱크 준비` — 새 리마인더 문구가 **복원된 일정에서 파생**됐습니다. 즉 기억이 다음 행동을 실제로 바꿨습니다(`ledger len 6` 으로 누적).
- 여기서는 ledger 한정의 최소 저장/복원만 보여줍니다. **전체 세션 스냅샷(ledger+artifact)·resume 은 선수 트랙 [Track 05](../track05_memory_and_long_context/05_memory_and_long_context_lab.ipynb)**, 라우팅까지 갖춘 **전체 개인 비서 구현은 [`implementations/hermes-agent`](../../implementations/hermes-agent)** 를 참고하세요 — 이 노트북은 그 **최소 골격**판입니다.


## Session 3. 패키지


### Session 3-1. 패키지

**하는 일:** 회귀 점수를 출력하고, 회귀·제출용 패키지 JSON을 저장합니다.

**정상:** `regression n=22 | M1=0.57 M6=0.50 M9(stub)=0.53`과 `saved .../_out/06/capstone_package.json` 두 줄이 출력됩니다(두 번째는 저장 경로, 절대경로).

**의미:** 캡스톤 제출·회귀용 결과를 화면으로 확인하고 한 파일로 묶습니다.

In [ ]:
regression = regression_m1_m6_m9(load_capstone_golden("06"))
# (en) Surface the static metric-demo numbers inline (metric MECHANICS, not the agent's score).
# (kr) 정적 메트릭 데모 수치를 화면에 보여준다(에이전트 성능이 아니라 메트릭 동작 예시).
print(f"regression n={regression['n']} | M1={regression['M1_mean']:.2f} M6={regression['M6_loose_mean']:.2f} M9(stub)={regression['M9_mean']:.2f}")
pkg_path = save_package("06", {
    "ledger_entries": len(ledger2),
    "ledger_path": "recipes/track10_ax_capstones/_out/06/ledger.jsonl",
    "skills": ["calendar", "reminder"],
    "hermes_ref": "implementations/hermes-agent",
    "regression": regression,
    "session_trace": [
        {"event": "ledger_append", "n": len(ledger)},
        {"event": "ledger_resume", "n": len(ledger2)},
    ],
})


**출력 해석:** `regression n=22 | M1=0.57 M6=0.50 M9(stub)=0.53`과 `saved .../_out/06/capstone_package.json` 두 줄이 출력되면 골든 회귀·ledger·skills·SLOSpec 이 한 파일로 묶여 캡스톤이 마감된 것입니다.

- `regression` 수치는 공통 골든 `all` 22행을 채점하는 **메트릭 동작 데모**일 뿐 라이브 에이전트 성능이 아닙니다(`06` 전용 행은 아직 없음): `M1=0.57` = `expected_answer`가 있는 7행 중 4건 정확 일치, `M6=0.50` = `required_keys`가 있는 6행 중 3건 키 충족, `M9(stub)=0.53` = `grounding_context`가 있는 5행에 대한 `LengthRatioJudge` 길이비 근사(충실도 아님, 테스트 전용 스텁).
- 저장 경로는 `TRACK10/_out/06/` 절대경로라 Jupyter 를 어느 디렉터리에서 띄우든 같은 위치에 저장됩니다 — 상대 `_out/`의 CWD 의존 문제를 피한 설계입니다.
- `ledger_entries: 6`(Session 2-2의 저장·복원·재활용까지 마친 최종 ledger)·`ledger_path`(저장된 JSONL 경로)·`skills: [calendar, reminder]`·`hermes_ref`가 함께 기록되어, 다시 실행해도 같은 입력이면 같은 패키지가 나오는 회귀 가능한 산출물이 됩니다.

## 마무리

이 캡스톤에서는 `calendar`·`reminder` 두 skill을 하나의 레지스트리에 등록하고, 질의와 결과를 `InMemoryLedger`에 기록했습니다. 이어 ledger를 `_out/06/ledger.jsonl`로 저장·복원한 뒤 복원된 일정을 다음 리마인더에 재활용하고, 전체 결과를 `_out/06/capstone_package.json`으로 저장했습니다.

**핵심 정리**
- **Ledger:** 질의와 도구 결과를 append-only로 쌓고, `as_list()`와 JSONL 저장·복원으로 세션을 넘어 이어갑니다.
- **Skills:** 함수 스키마와 실행 콜백을 단일 레지스트리에 등록해 `reg.execute(name, args)`로 호출합니다.
- **재활용:** 복원된 캘린더 결과가 다음 리마인더 입력으로 쓰여 기억이 실제 행동을 바꿉니다.
- **패키지:** ledger, skills, trace, SLO, 정적 메트릭을 한 JSON으로 묶습니다.

**한계**
- `calendar`/`reminder`는 실제 외부 서비스가 아니라 고정 반환값입니다.
- 어떤 skill을 부를지는 코드가 직접 지정합니다. 자동 라우팅은 Track 06 영역입니다.
- 저장/복원은 ledger 한정 최소판입니다. 전체 세션 스냅샷은 Track 05, 전체 개인 비서 구현은 `implementations/hermes-agent`를 참고하세요.
- 정적 메트릭은 fixture 채점 데모이며 라이브 에이전트 성능이 아닙니다.

**다음:** `10_07` 프로덕션 하네스로 캡스톤들을 운영 수준으로 묶습니다.

## 체크포인트

- [ ] Session 2-1 단일 레지스트리 등록 + skill 실행 + ledger 재생 확인
- [ ] Session 2-2 ledger 저장 → 복원 → 재활용 확인
- [ ] Session 3 골든 회귀 실행 + `_out/06/capstone_package.json` 저장
